In [1]:
from openai import OpenAI
import openai
import os

api_key = os.getenv('OPENAI_API_KEY')
openai.api_key = api_key

- chat gpt: 모델학습 이후 프롬프트에 대해 자동 웹서치
- open ai api :모델 학습 부분까지만 답

In [2]:
client = OpenAI( )
response = client.chat.completions.create( 
    model='gpt-3.5-turbo',
    messages=[{'role':'user','content':'서울 현재 기온 알려줄래'}] )

print( response.choices[0].message.content)

죄송해요, 제가 실시간으로 기상 정보를 제공해드릴 수는 없어요. 하지만 기상청 웹사이트나 기상 앱을 통해 현재 서울의 기온을 확인하실 수 있어요. 현재 날씨가 매우 추운 것으로 알려져 있으니 외출 시 따뜻하게 입고 다니시길 권해드려요.


### 위도와 경도를 통해 날씨 정보를 얻는 함수

In [3]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    # print('data:',data['current'] )
    return data['current']['temperature_2m']
    # return data['current']['temperature_2m'], data['current']['time']

In [4]:
result = get_weather( 37.5642135, 127.0016985)
result

-8.7

In [5]:
wtool= [{
    "type": "function", #함수타입이다
    "function":{
        "name":"get_weather", # gpt 가 호출할 함수명수명
        "description": "Get current temperature for provided coordinates in celsius.", #GPT가 언제 이 함수를 써야 하는지를 이해하는 데 도움을 주는 프롬프트
        "parameters":{
            "type":"object",
            "properties":{
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required":["latitude", "longitude"],
            "additionalProperties": False  #추가 인자는 없다.
        },
        "strict": True  # 정의된 schema 외에는 인자를 허용하지 않는다.
    }
}]

In [6]:
import json

client = OpenAI( )
messages=[{'role':'user','content':'부산 현재 기온 알려줄래'}]
response = client.chat.completions.create( 
    model='gpt-3.5-turbo',
    messages=messages,
    tools=wtool, tool_choice="auto" ) #날씨가 정보를 얻는 함수가 있다. 함수 호출을 위해 latitude, longitude를 필수로

try:
    print( response )
    print( response.choices[0].message.tool_calls[0].function.arguments )
    args = json.loads( response.choices[0].message.tool_calls[0].function.arguments )
    c = get_weather( args['latitude'], args['longitude'])
    print('현재기온', c)

    messages.append( {'role':'function','name':'get_weather','content':json.dumps(c)} )

    response =client.chat.completions.create( model='gpt-4o-mini', messages= messages )
    print( response.choices[0].message.content)

except Exception as err:
    print( response.choices[0].message.content)

ChatCompletion(id='chatcmpl-CxizRA7mcDbPHyvbjHweL2EGhwxpn', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Cn2Oz7GTYFyqG35wv0hNzZsl', function=Function(arguments='{"latitude":35.1796,"longitude":129.0756}', name='get_weather'), type='function')]))], created=1768350329, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=24, prompt_tokens=66, total_tokens=90, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
{"latitude":35.1796,"longitude":129.0756}
현재기온 -2.5
현재 부산의 기온은 -2.5도입니다. 날씨에 유의하시기 바랍니다!


### 여러개의 함수정보

In [7]:
wtool= [{
    "type": "function", #함수타입이다
    "function":{
        "name":"get_weather", # gpt 가 호출할 함수명수명
        "description": "Get current temperature for provided coordinates in celsius.", #GPT가 언제 이 함수를 써야 하는지를 이해하는 데 도움을 주는 프롬프트
        "parameters":{
            "type":"object",
            "properties":{
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required":["latitude", "longitude"],
            "additionalProperties": False  #추가 인자는 없다.
        },
        "strict": True  # 정의된 schema 외에는 인자를 허용하지 않는다.
    }
},
{
    'type':'function',
    'function':{
        'name':'add_number',
        'description':'두 숫자의 합을 계산합니다.',
        'parameters':{
            'type':'object',
            'properties':{
                'num1':{'type':'number'},
                'num2':{'type':'number'}
            },
            'required':['num1', 'num2']
        }
    }
}]

In [8]:
def add_number( num1, num2):
    print( 'call', num1, num2)
    return {'합':num1+num2}

In [9]:
sumtool = [{
    'type':'function',
    'function':{
        'name':'add_number',
        'description':'두 숫자의 합을 계산합니다.',
        'parameters':{
            'type':'object',
            'properties':{
                'num1':{'type':'number'},
                'num2':{'type':'number'}
            },
            'required':['num1', 'num2']
        }
    }
}]

In [ ]:
messages=[{'role':'user','content':'5와 3을 더해 주세요'}]
client = OpenAI( )
response = client.chat.completions.create( 
    model='gpt-3.5-turbo',
    messages= messages,
    tools=sumtool )

print( response.choices[0].message.tool_calls[0].function.arguments )
args = json.loads( response.choices[0].message.tool_calls[0].function.arguments )
result = add_number( args['num1'], args['num2'] )
# result {'합':8}
messages.append( {'role':'function','name':'add_number','content':json.dumps(result)} )
# messages=[{'role':'user','content':'5와 3을 더해 주세요'},
# {'role':'function','name':'add_number','content':"{'합':8}" } ]
response =client.chat.completions.create( model='gpt-3.5-turbo',messages= messages )
print( response.choices[0].message.content)

{"num1":5,"num2":3}
call 5 3
5와 3을 더한 값은 8입니다.


In [1]:
%pip install pytz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


https://docs.python.org/3.6/library/datetime.html

In [3]:
import pytz
from datetime import datetime
tz = pytz.timezone( 'Asia/Seoul')
# tz = pytz.timezone( 'Asia/Tokyo')
local_time = datetime.now( tz )
local_time.strftime( '시간:%Y-%m-%d %H:%M:%S')

'시간:2026-01-14 09:48:31'

In [5]:
def get_time( city ):
    timezones={
        '서울':'Asia/Seoul',
        '도쿄':'Asia/Tokyo',
        '뉴욕':'America/New_York',
        '런던':'Europe/London'
    }
    try:
        tz = pytz.timezone( timezones[city] )
        local_time = datetime.now( tz )
        return local_time.strftime( '시간:%Y-%m-%d %H:%M:%S')
    except Exception as err:
        return "현재 정보로는 시간을 알수 없습니다."

In [6]:
get_time('서울')

'시간:2026-01-14 09:48:57'

In [7]:
get_time('뉴욕')

'시간:2026-01-13 19:48:59'

### 5분 퀴즈....

# 3개숫자의 가장큰값을 구하는 함수를 만들고
# 20 과 30과 10 중의 가장큰수는 뭐니?


In [16]:
def compare_number(num1, num2, num3):
    mx = num1 if num1>num2 else num2
    mx = mx if mx>num3 else num3
    return mx

In [17]:
comptool = [{
    'type':'function',
    'function':{
        'name':'compare_number',
        'description':'세 숫자를 비교하여 큰수를 찾습니다.',
            'parameters':{
            'type':'object',
            'properties':{
            'num1':{'type':'number'},
            'num2':{'type':'number'},
            'num3':{'type':'number'}
            },
        'required':['num1', 'num2','num3']
        }
    }
}]
messages = [{'role':'user','content':'20과 30과 10 중에서 가장 큰수는?'}]
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=comptool)


args = response.choices[0].message.tool_calls[0].function.arguments
args = json.loads(args)
result = compare_number(args['num1'],args['num2'],args['num3'])

messages.append( {'role':'function','name':'compare_number','content':json.dumps(result)} )

response =client.chat.completions.create( model='gpt-4o-mini',messages= messages )
print( response.choices[0].message.content)

가장 큰 수는 30입니다.
